# Expanded-label model: analyze and render seven-point videos

This notebook uses the completed `mobile_worklabels_20260927_210240` run.
It runs video inference if prediction H5 files are missing, then renders clean
seven-point labeled videos at `pcutoff=0.2`.



In [ ]:
%pip -q install --pre deeplabcut



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path, PurePosixPath
import shutil
import zipfile

import deeplabcut
import pandas as pd
import yaml
from deeplabcut.utils import auxiliaryfunctions

ROOT = Path('/content/drive/MyDrive/sp1DLC')
RUN = ROOT / 'Colab_Training_Runs/mobile_worklabels_20260927_210240'
SOURCE_ZIP = ROOT / 'all_32_videos.zip'
PREDICTIONS = RUN / 'videos'
OUTPUT = RUN / 'Seven_Point_Videos_p20'
WORK = Path('/content/worklabels_video_render')
VIEW = Path('/content/worklabels_model_view')
CONFIG = VIEW / 'config.yaml'

FOCUS = ['nose', 'left_ear', 'right_ear', 'spine_mid', 'left_hip', 'right_hip', 'tail_base']
SKELETON = [
    ['nose', 'left_ear'], ['nose', 'right_ear'],
    ['left_ear', 'spine_mid'], ['right_ear', 'spine_mid'],
    ['spine_mid', 'left_hip'], ['spine_mid', 'right_hip'],
    ['left_hip', 'tail_base'], ['right_hip', 'tail_base'],
]

assert SOURCE_ZIP.is_file(), f'Missing original video archive: {SOURCE_ZIP}'
assert (RUN / 'config.yaml').is_file(), f'Missing run config: {RUN / "config.yaml"}'
assert list((RUN / 'dlc-models-pytorch').rglob('snapshot-best-110.pt')), 'Best checkpoint is missing.'

WORK.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
PREDICTIONS.mkdir(exist_ok=True)

# Extract only Test videos that match the DLC project.
with (RUN / 'config.yaml').open(encoding='utf-8') as handle:
    original_cfg = yaml.safe_load(handle)
expected = {Path(str(path).replace('\\', '/')).name for path in original_cfg['video_sets']}
assert len(expected) == 31, f'Expected 31 video_set entries, found {len(expected)}'

VIDEO_ZIP = Path('/content/all_32_videos.zip')
if not VIDEO_ZIP.is_file() or VIDEO_ZIP.stat().st_size != SOURCE_ZIP.stat().st_size:
    shutil.copy2(SOURCE_ZIP, VIDEO_ZIP)

videos = {}
with zipfile.ZipFile(VIDEO_ZIP) as archive:
    assert archive.testzip() is None, 'Original video ZIP failed its integrity check.'
    for info in archive.infolist():
        name = PurePosixPath(info.filename.replace('\\', '/')).name
        if name in expected:
            target = WORK / name
            if not target.is_file() or target.stat().st_size != info.file_size:
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
            videos[name] = target
assert set(videos) == expected, f'Original videos missing: {sorted(expected - set(videos))}'

# Create a render/analyze config whose project_path points at a light temporary model view.
VIEW.mkdir(exist_ok=True)
models_link = VIEW / 'dlc-models-pytorch'
if not models_link.is_symlink():
    assert not models_link.exists(), f'Unexpected model folder: {models_link}'
    models_link.symlink_to(RUN / 'dlc-models-pytorch', target_is_directory=True)

cfg = dict(original_cfg)
cfg['project_path'] = str(VIEW)
cfg['video_sets'] = {str(WORK / name): value for name, value in original_cfg['video_sets'].items()}
cfg['snapshotindex'] = -1
cfg['dotsize'] = 3
cfg['skeleton'] = SKELETON
with CONFIG.open('w', encoding='utf-8') as output:
    yaml.safe_dump(cfg, output, sort_keys=False)

scorer, _ = auxiliaryfunctions.get_scorer_name(cfg, 1, cfg['TrainingFraction'][0], snapshot_index='best')
print('Run:', RUN)
print('Videos:', len(videos))
print('Scorer:', scorer)
print('Prediction folder:', PREDICTIONS)
print('Video output folder:', OUTPUT)



In [ ]:
# Run inference only when the expected prediction files are not already present.
existing = sorted(PREDICTIONS.glob(f'*{scorer}.h5'))
if len(existing) != len(videos):
    print(f'Found {len(existing)} prediction files; running inference for {len(videos)} videos.')
    deeplabcut.analyze_videos(
        str(CONFIG),
        [str(path) for _, path in sorted(videos.items())],
        shuffle=1,
        videotype='.mp4',
        destfolder=str(PREDICTIONS),
        snapshot_index='best',
        save_as_csv=True,
    )
else:
    print('Prediction files already present; skipping video inference.')

h5_files = sorted(PREDICTIONS.glob(f'*{scorer}.h5'))
assert len(h5_files) == len(videos), f'Expected {len(videos)} prediction H5 files, found {len(h5_files)}'
print('Ready to render from', len(h5_files), 'prediction files')



In [ ]:
# Copy seven-column prediction H5 files into the working folder so the renderer
# cannot draw non-focus bodyparts.
for h5 in h5_files:
    meta = h5.with_name(h5.stem + '_meta.pickle')
    assert meta.is_file(), f'Missing prediction metadata: {meta.name}'
    predictions = pd.read_hdf(h5)
    present = list(predictions.columns.get_level_values('bodyparts').unique())
    assert all(point in present for point in FOCUS), f'{h5.name} is missing a focus point.'
    filtered = predictions.loc[:, predictions.columns.get_level_values('bodyparts').isin(FOCUS)]
    assert list(filtered.columns.get_level_values('bodyparts').unique()) == FOCUS
    filtered.to_hdf(WORK / h5.name, key='df_with_missing', mode='w')
    shutil.copy2(meta, WORK / meta.name)

render_cfg = dict(cfg)
render_cfg['bodyparts'] = FOCUS
with CONFIG.open('w', encoding='utf-8') as output:
    yaml.safe_dump(render_cfg, output, sort_keys=False)

results = deeplabcut.create_labeled_video(
    str(CONFIG),
    [str(path) for _, path in sorted(videos.items())],
    destfolder=str(WORK),
    shuffle=1,
    displayedbodyparts=FOCUS,
    snapshot_index='best',
    pcutoff=0.2,
    draw_skeleton=True,
    trailpoints=0,
    fastmode=True,
    plot_bboxes=False,
    max_workers=1,
    overwrite=True,
)
assert len(results) == len(videos) and all(results), 'One or more videos failed to render.'

made = sorted(WORK.glob(f'*{scorer}_p20_labeled.mp4'))
assert len(made) == len(videos), f'Expected {len(videos)} labeled videos, found {len(made)}'
for video in made:
    shutil.copy2(video, OUTPUT / video.name)

print('Finished:', len(made), 'seven-point videos in', OUTPUT)



In [ ]:
from IPython.display import Video, display

sample = OUTPUT / f'Test 30{scorer}_p20_labeled.mp4'
if sample.is_file():
    display(Video(str(sample), embed=True, width=720))
else:
    print('Sample not found:', sample)
